## Here we will use the unit made in last notebook to make a simple Neural Network

#### I added the unit class code into the backpropogation.py file

In [1]:
from backpropogation import unit
import random

### First is Neuron
#### Neuron has n weights and 1 bias parameter

In [2]:
class Neuron:
    def __init__(self, n_inputs, activation='relu'):
        self.weights = [unit(random.uniform(-1, 1), symbol = f'w{i}')for i in range(n_inputs)]
        self.bias = unit(0.0, symbol = 'b')
        self.output = None
        self.activation = activation
        self._params = self.weights + [self.bias]
        
    def forward(self, inputs):
        value = sum((weight*inputs[i] for i,weight in enumerate(self.weights)), self.bias)
        if self.activation == 'tanh':
            output_value = value.tanh()
        elif self.activation == 'relu':
            output_value = value.ReLU()
        else:
            output_value = value
        self.output = output_value
        return output_value

    def parameters(self):
        return self._params

    def update_weights(self, alpha):
        for param in self._params:
            param.data -= alpha * param.grad

    def zero_gradient(self):
        for param in self._params:
            param.grad = 0
        

While using above code I got 1 thing which was when you run this somethimes all the grad is 0 and output is also 0
Why is this happening?
..
ReLU dead neuron problem
If the total output is negative or 0 then relu value will become 0
Now as relu output is 0 so the input to relu is dependent on relu output as 
```
def ReLU(self):
        out = unit(self.data if self.data > 0 else 0, [self], symbol = f"(ReLU({self.symbol}))")
        self.ref_count += 1
        def func():
            self.grad += out.grad if self.data > 0 else 0
        out._backprop = func
        return out
```
L = ReLU(x)
x value is negative
out in code L above will be 0
self.grad which is x.grad will be 0 as self.data < 0
so x.grad will be 0
so all the value before x will also have grad as 0

### Second part is Layer

#### For a Layer there will have n_in inputs, n_out outputs

In [3]:
class Layer:
    def __init__(self, n_in, n_out, activation='relu'):
        self.neurons = [Neuron(n_in, activation=activation) for _ in range(n_out)]

    # inputs will be a vector of size n_in
    def forward(self, inputs):
        output = [self.neurons[i].forward(inputs) for i in range(len(self.neurons))]
        return output

    def parameters(self):
        params = []
        for neuron in self.neurons:
            params += neuron.parameters()
        return params

    def update_weights(self, alpha):
        for neuron in self.neurons:
            neuron.update_weights(alpha)

    def zero_gradient(self):
        for neuron in self.neurons:
            neuron.zero_gradient()
        
            


In [4]:
class MLP:
    def __init__(self, layers_size_list):
        n = len(layers_size_list)
        self.layers = [Layer(layers_size_list[i-1], layers_size_list[i]) if i != n-1 else Layer(layers_size_list[i-1], layers_size_list[i], 'linear') for i in range(1,n)]
        self.output = None
        
    def forward(self, inputs):
        output_val = inputs
        for layer in self.layers:
            output_val = layer.forward(output_val)
        self.output = output_val
        return output_val

    def parameters(self):
        params = []
        for layer in self.layers:
            params += layer.parameters()
        return params

    def update_weights(self, alpha):
        for layer in self.layers:
            layer.update_weights(alpha)

    def zero_gradient(self):
        for layer in self.layers:
            layer.zero_gradient()
        

In [5]:
# XOR dataset
inputs = [
    [unit(0.0), unit(0.0)],
    [unit(0.0), unit(1.0)],
    [unit(1.0), unit(0.0)],
    [unit(1.0), unit(1.0)],
]
targets = [0.0, 1.0, 1.0, 0.0]

In [6]:
NN = MLP([2,4,1])

In [7]:
n = len(inputs)
for epoch in range(500):
    y_pred = []
    for i in range(n):
        y_pred.append(NN.forward(inputs[i]))
    
    Loss = 0.0
    for i in range(n):
        Loss += (y_pred[i][0] - targets[i])**2
    Loss /= n
    
    if epoch % 50 == 0:
        print(f"epoch {epoch}, loss {Loss.data:.4f}")
    
    Loss.backward()
    NN.update_weights(0.01)
    NN.zero_gradient()

epoch 0, loss 0.7278
epoch 50, loss 0.0710
epoch 100, loss 0.0475
epoch 150, loss 0.0351
epoch 200, loss 0.0290
epoch 250, loss 0.0241
epoch 300, loss 0.0201
epoch 350, loss 0.0167
epoch 400, loss 0.0138
epoch 450, loss 0.0114


In [8]:
for i in range(n):
    print(y_pred[i][0].data)

0.17030590076463756
0.9225733360874389
0.95065683278811
0.018389112433568985


The code I wrote for Neuron, Layer and MLP is correct and working fine right now, I checked Andrej Karpathy's version and thought as I implemented the backward() pass with queue and indegree (it is topo sort only but I remembered I used to implement this way), here also he used all the parameters and added a for loop, mine is far better, but here I am commiting a mistake, a huge mistake of overconfidence and doubting Andrej :). The mistake is strongly coupling model and optimizer, see I have added the logic of update weights in the model.
It should never be the case, why because our model is just an architecture, it should just know parameters and how a forward pass works, nothing less and nothing more. 
We have seperate function for seperate entities - model, loss, optimizer in the training loop. If there are other contents as well Idk.

How a Training loop works
1. First thing in our training loop is forward pass, which should be the property of out model
2. Second comes calculating the loss function, and doing loss.backward() which calculates the gradient of loss w.r.t. all parameters. No update to the parameter only finding gradients.
3. Third comes the part which is optimizer, which does 2 things
   1. Doing optimizer.step() which is nothing just updating all the parameters - now the mistake we made we implemented the update_weights() (which is gradient descent actually) in the model itself
   2. optimizer.zero_grad() which is making all the gradients as zero, this is very useful as when we were finding grad, a same parameter could be used to calculate many outputs, so we used ```self.grad += out.grad * 1``` this is for addition function. We are accumulating gradients which is needed in a single iteration. But for the other iteration, it should start from 0.

These are the learning by implementing wrong code.

# Day 3

In [19]:
class Neuron:
    def __init__(self, n_inputs, activation='relu'):
        self.weights = [unit(random.uniform(-1, 1), symbol = f'w{i}')for i in range(n_inputs)]
        self.bias = unit(0.0, symbol = 'b')
        self.output = None
        self.activation = activation
        self._params = self.weights + [self.bias]
        
    def forward(self, inputs):
        value = sum((weight*inputs[i] for i,weight in enumerate(self.weights)), self.bias)
        if self.activation == 'tanh':
            output_value = value.tanh()
        elif self.activation == 'relu':
            output_value = value.ReLU()
        else:
            output_value = value
        self.output = output_value
        return output_value

    def parameters(self):
        return self._params

class Layer:
    def __init__(self, n_in, n_out, activation='relu'):
        self.neurons = [Neuron(n_in, activation=activation) for _ in range(n_out)]

    # inputs will be a vector of size n_in
    def forward(self, inputs):
        output = [self.neurons[i].forward(inputs) for i in range(len(self.neurons))]
        return output

    def parameters(self):
        params = []
        for neuron in self.neurons:
            params += neuron.parameters()
        return params

        
            
class MLP:
    def __init__(self, layers_size_list):
        n = len(layers_size_list)
        self.layers = [Layer(layers_size_list[i-1], layers_size_list[i]) if i != n-1 else Layer(layers_size_list[i-1], layers_size_list[i], 'linear') for i in range(1,n)]
        self.output = None
        
    def forward(self, inputs):
        output_val = inputs
        for layer in self.layers:
            output_val = layer.forward(output_val)
        self.output = output_val
        return output_val

    def parameters(self):
        params = []
        for layer in self.layers:
            params += layer.parameters()
        return params
        
        

In [20]:
# impelemting one type of optimizer
# SGD - Stochastic Gradient Descent
# Stochastic Gradient Descent is an optimization algorithm used in machine learning,
# especially for large datasets, that updates model parameters efficiently using small batches or single samples.
class SGD:
    def __init__(self, parameters, learning_rate):
        self.parameters = parameters
        self.learning_rate = learning_rate

    def step(self):
        for para in self.parameters:
            para.data -= self.learning_rate * para.grad 

    def zero_grad(self):
        for para in self.parameters:
            para.grad = 0
        

In [49]:
# XOR dataset
inputs = [
    [unit(0.0), unit(0.0)],
    [unit(0.0), unit(1.0)],
    [unit(1.0), unit(0.0)],
    [unit(1.0), unit(1.0)],
]
targets = [0.0, 1.0, 1.0, 0.0]

In [50]:
NN = MLP([2,4,1])

In [51]:
optimizer = SGD(NN.parameters(), 0.01)

In [55]:
n = len(inputs)
for epoch in range(100):
    y_pred = [NN.forward(inputs[i]) for i in range(n)]
    for params in NN.parameters():
        if epoch % 10 == 0:
            print(params.data, params.grad, params.ref_count)

    Loss = 0.0
    for i in range(n):
        Loss += (y_pred[i][0] - targets[i])**2
    Loss /= n
    # for params in Loss.children:
    #     if epoch % 10 == 0:
    #         print(params.data, params.grad, params.ref_count)
    Loss.backward()
    if epoch % 10 == 0:
            print("x"*80)
    
    for params in NN.parameters():
        if epoch % 10 == 0:
            print(params.data, params.grad, params.ref_count)

    if epoch % 50 == 0:
        print(f"epoch {epoch}, loss {Loss.data:.4f}")

    optimizer.step()
    if epoch % 10 != 0:
        optimizer.zero_grad()

-0.17031286527425626 0 4
0.5499650487226114 0 4
0.003016877890138165 0 4
-0.5013532760512962 0 4
-0.5581607049693889 0 4
0.0 0 4
-0.5614733515477786 0 4
-0.43407739203885365 0 4
0.0 0 4
0.8488084297492133 0 4
-0.04204000484937029 0 4
-0.002326999584629432 0 4
0.19496994583302946 0 4
-0.5469763819804407 0 4
0.967244889029361 0 4
0.5757646094682881 0 4
0.0431412712484572 0 4
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
-0.17031286527425626 0.056630886677969394 0
0.5499650487226114 -0.02613813223501435 0
0.003016877890138165 -0.021875165864127338 0
-0.5013532760512962 0.0 0
-0.5581607049693889 0.0 0
0.0 0.0 0
-0.5614733515477786 0.0 0
-0.43407739203885365 0.0 0
0.0 0.0 0
0.8488084297492133 0.03208000969142044 0
-0.04204000484937029 0.1672363410302605 0
-0.002326999584629432 0.03208000969142044 0
0.19496994583302946 -0.12353710384758504 0
-0.5469763819804407 0.0 0
0.967244889029361 0.0 0
0.5757646094682881 0.03495267955765627 0
0.0431412712484572 -0.3469

In [56]:
for params in NN.parameters():
    print(params.data, params.grad, params.ref_count)

-0.25288984687247573 0 0
0.5619436640740956 0 0
-0.00032374839743792795 0 0
-0.5013532760512962 0 0
-0.5581607049693889 0 0
0.0 0 0
-0.5614733515477786 0 0
-0.43407739203885365 0 0
0.0 0 0
0.7880086859629288 0 0
-0.22615561283840546 0 0
-0.06312674337091415 0 0
0.29377742236087 0 0
-0.5469763819804407 0 0
0.967244889029361 0 0
0.5337198640678519 0 0
0.23159100769535412 0 0


In [57]:
inputs[0][0].ref_count

0

In [58]:
Loss.data

0.2279156903743654